# 10.4 Restricted Boltzmann Machines and Deep Belief Nets

RBMs define probability through energy; deep belief nets stack those energies into hierarchical latent features.

Log-linear models, sigmoid probabilities, and Markov chains are the prerequisites. An RBM uses an energy score and conditional Bernoulli Gibbs updates to sample binary visible and hidden units.

Save a copy to Drive to edit. This notebook is CPU-only, seeded, and designed to be inspected before you run any cell.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

SEED = 1007
rng = np.random.default_rng(SEED)
random.seed(SEED)
np.random.seed(SEED)


def sigmoid(a):
    return 1.0 / (1.0 + np.exp(-a))


def stable_log(x):
    return np.log(np.clip(x, 1e-8, 1.0))


def standardize(X):
    X = np.asarray(X, dtype=float)
    center = X.mean(axis=0, keepdims=True)
    scale = X.std(axis=0, keepdims=True)
    scale = np.where(scale < 1e-6, 1.0, scale)
    return (X - center) / scale


def pca_project_reconstruct(X, latent_dim, shrink=1.0):
    X = np.asarray(X, dtype=float)
    latent_dim = int(min(latent_dim, X.shape[0] - 1, X.shape[1]))
    latent_dim = max(latent_dim, 1)
    model = PCA(n_components=latent_dim, random_state=SEED)
    Z = model.fit_transform(X)
    Z = Z * shrink
    X_hat = model.inverse_transform(Z)
    return model, Z, X_hat


def reconstruction_error(X, X_hat):
    return float(0.5 * np.mean(np.sum((X - X_hat) ** 2, axis=1)))


def rbf_two_sample_distance(A, B, gamma=None):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    both = np.vstack([A, B])
    if gamma is None:
        d = pairwise_distances(both[: min(len(both), 80)])
        positive = d[d > 0]
        width = np.median(positive) if positive.size else 1.0
        gamma = 1.0 / (2.0 * width ** 2 + 1e-8)
    Kaa = np.exp(-gamma * pairwise_distances(A, A) ** 2).mean()
    Kbb = np.exp(-gamma * pairwise_distances(B, B) ** 2).mean()
    Kab = np.exp(-gamma * pairwise_distances(A, B) ** 2).mean()
    return float(Kaa + Kbb - 2.0 * Kab)


def sample_latent_decode(model, Z, n_samples, noise_scale=0.75):
    mu = Z.mean(axis=0)
    sigma = Z.std(axis=0) + 1e-6
    Z_new = rng.normal(mu, noise_scale * sigma, size=(n_samples, Z.shape[1]))
    return model.inverse_transform(Z_new)


def make_digit_hard_set(limit=240):
    digits = load_digits()
    X = digits.data.astype(float) / 16.0
    y = digits.target
    X = X[:limit]
    y = y[:limit]
    rows = []
    for i, row in enumerate(X):
        image = row.reshape(8, 8)
        shift = (i % 3) - 1
        moved = np.roll(image, shift=shift, axis=1)
        nuisance = 0.20 * np.sin(np.linspace(0.0, np.pi, 64) + 0.3 * y[i])
        noisy = np.clip(moved.reshape(-1) + nuisance + rng.normal(0.0, 0.06, 64), 0.0, 1.0)
        rows.append(noisy)
    hard = np.asarray(rows)
    interactions = hard[:, :16] * hard[:, 16:32]
    return np.hstack([hard, interactions])


def build_f9_ladder():
    X1 = rng.normal(0.0, 1.0, size=(160, 1))
    X2, y2 = make_moons(n_samples=180, noise=0.06, random_state=SEED)
    means = np.array([[-2.0, 0.0], [1.5, 1.3], [1.2, -1.4]])
    parts = []
    labels = []
    for k, mean in enumerate(means):
        cov = np.array([[0.10 + 0.05 * k, 0.03], [0.03, 0.18]])
        draw = rng.multivariate_normal(mean, cov, size=70)
        parts.append(draw)
        labels.extend([k] * len(draw))
    X3 = np.vstack(parts)
    y3 = np.asarray(labels)
    digits = load_digits()
    X4 = digits.data.astype(float) / 16.0
    y4 = digits.target
    X4 = X4[:240]
    y4 = y4[:240]
    X5 = make_digit_hard_set(limit=240)
    y5 = y4.copy()
    return [
        {"rung": "D1", "name": "1-D Gaussian", "X": standardize(X1), "y": None, "image_shape": None},
        {"rung": "D2", "name": "2-D two moons", "X": standardize(X2), "y": y2, "image_shape": None},
        {"rung": "D3", "name": "3-component mixture", "X": standardize(X3), "y": y3, "image_shape": None},
        {"rung": "D4", "name": "sklearn digits", "X": standardize(X4), "y": y4, "image_shape": (8, 8)},
        {"rung": "D5", "name": "noisy shifted digits with interactions", "X": standardize(X5), "y": y5, "image_shape": (8, 8)},
    ]


def preview_ladder(ladder):
    for item in ladder:
        X = item["X"]
        y = item["y"]
        classes = "none" if y is None else len(np.unique(y))
        print(f"{item['rung']} | {item['name']} | shape={X.shape} | classes={classes}")
        print(np.round(X[:2, : min(6, X.shape[1])], 3))


def first_two_dimensions(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 1:
        return np.column_stack([X[:, 0], np.zeros(len(X))])
    if X.shape[1] == 2:
        return X
    model = PCA(n_components=2, random_state=SEED)
    return model.fit_transform(X)


def panel_axis(ax, X, title, image_shape=None):
    X = np.asarray(X, dtype=float)
    if image_shape is not None and X.shape[1] >= image_shape[0] * image_shape[1]:
        side = image_shape[0] * image_shape[1]
        mosaic = X[:16, :side].reshape(16, image_shape[0], image_shape[1])
        rows = []
        for start in range(0, 16, 4):
            rows.append(np.hstack(mosaic[start:start + 4]))
        ax.imshow(np.vstack(rows), cmap="gray")
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        xy = first_two_dimensions(X[:120])
        ax.scatter(xy[:, 0], xy[:, 1], s=10, alpha=0.75)
    ax.set_title(title, fontsize=9)


def plot_generated_panels(results, metric_name):
    fig, axes = plt.subplots(2, len(results), figsize=(3.0 * len(results), 5.0))
    for j, result in enumerate(results):
        panel_axis(axes[0, j], result["generated"], result["rung"], result.get("image_shape"))
        axes[1, j].bar([0], [result["metric"]])
        axes[1, j].set_xticks([0])
        axes[1, j].set_xticklabels([metric_name], rotation=30, ha="right")
        axes[1, j].set_title(f"{result['metric']:.3f}", fontsize=9)
    fig.tight_layout()
    plt.show()
    plt.figure(figsize=(6, 3))
    plt.plot([r["rung"] for r in results], [r["metric"] for r in results], marker="o")
    plt.ylabel(metric_name)
    plt.xlabel("dataset complexity")
    plt.title(f"{metric_name} vs complexity")
    plt.grid(True, alpha=0.3)
    plt.show()


## The concept, built once: RBM energy and probability

The lesson formula is $E(v,h)=-b^\top v-c^\top h-v^\top Wh$ and $p(v,h)=\exp(-E(v,h))/Z$. We assert the exact energy and unnormalized weight.

In [ ]:
def rbm_energy_and_gibbs(v, h, b, c, W):
    energy = -float(b @ v + c @ h + v @ W @ h)
    hidden_prob = sigmoid(c + v @ W)
    visible_prob = sigmoid(b + W @ h)
    return energy, hidden_prob, visible_prob

v = np.array([1.0, 0.0])
h = np.array([1.0, 1.0])
b = np.array([0.1, -0.2])
c = np.array([0.2, 0.1])
W = np.array([[0.2, 0.1], [0.0, 0.1]])
energy, hidden_prob, visible_prob = rbm_energy_and_gibbs(v, h, b, c, W)
weight = float(np.exp(-energy))
other_weight = 1.0
normalized = weight / (weight + other_weight)
print(round(energy, 4), round(weight, 3), round(normalized, 4), np.round(hidden_prob, 3))
assert round(energy, 4) == -0.7000
assert round(weight, 3) == 2.014
assert round(normalized, 4) == 0.6682

The reusable method binarizes each rung, initializes RBM weights from data correlations, performs Bernoulli hidden and visible Gibbs updates, and uses one-step reconstruction cross-entropy as an NLL proxy.

In [ ]:
def run_topic_model(item):
    X = item["X"]
    X01 = (X > np.median(X, axis=0, keepdims=True)).astype(float)
    hidden_dim = min(16, max(2, X01.shape[1] // 4))
    p = np.clip(X01.mean(axis=0), 0.05, 0.95)
    b = np.log(p / (1.0 - p))
    c = np.zeros(hidden_dim)
    W = rng.normal(0.0, 0.05, size=(X01.shape[1], hidden_dim))
    W += 0.02 * (X01.T @ rng.normal(0.0, 1.0, size=(X01.shape[0], hidden_dim))) / len(X01)
    ph = sigmoid(c + X01 @ W)
    h_sample = (rng.random(ph.shape) < ph).astype(float)
    pv = sigmoid(b + h_sample @ W.T)
    generated = pv[:40]
    metric = float(-np.mean(X01 * stable_log(pv) + (1.0 - X01) * stable_log(1.0 - pv)))
    return {
        "rung": item["rung"],
        "name": item["name"],
        "metric": metric,
        "generated": generated,
        "image_shape": item["image_shape"],
    }

## The D1-D5 generative ladder

Family F9 uses an inline ladder rather than a shared helper: D1 is a 1-D Gaussian, D2 is two moons, D3 is a mixture, D4 is bundled `sklearn.datasets.load_digits`, and D5 is a harder no-download real/synthetic digits set with shifts, nuisance variation, and feature interactions.

In [ ]:
ladder = build_f9_ladder()
preview_ladder(ladder)

In [ ]:
results = []
for item in ladder:
    result = run_topic_model(item)
    results.append(result)
    print(f"{result['rung']} {result['name']}: one_step_nll_proxy={result['metric']:.4f}")

In [ ]:
plot_generated_panels(results, "RBM NLL proxy")

## Pitfall on D5: short contrastive-divergence chains are not exact likelihood

CD-1 can leave negative samples near the data. Longer tiny chains are still approximate, but comparing the negative phase exposes bias.

In [ ]:
d5 = ladder[-1]
X = (d5["X"] > np.median(d5["X"], axis=0, keepdims=True)).astype(float)
hidden_dim = 12
b = np.zeros(X.shape[1])
c = np.zeros(hidden_dim)
W = rng.normal(0.0, 0.04, size=(X.shape[1], hidden_dim))

def gibbs_chain(v0, steps):
    v = v0.copy()
    for step in range(steps):
        ph = sigmoid(c + v @ W)
        h = (rng.random(ph.shape) < ph).astype(float)
        pv = sigmoid(b + h @ W.T)
        v = (rng.random(pv.shape) < pv).astype(float)
    return v

cd1 = gibbs_chain(X[:80], 1)
cd8 = gibbs_chain(X[:80], 8)
bias_1 = float(np.mean(np.abs(X[:80].mean(axis=0) - cd1.mean(axis=0))))
bias_8 = float(np.mean(np.abs(X[:80].mean(axis=0) - cd8.mean(axis=0))))
print(f"CD-1 negative phase distance={bias_1:.4f}")
print(f"CD-8 diagnostic distance={bias_8:.4f}")

## Evaluate it + practice

- Metric: track NLL proxy / one-step reconstruction error on every rung and compare against a no-skill baseline that samples noisy training examples.
- Sanity check: generated samples should match the rough support of D1-D3 before you trust D4-D5 panels.
- Ablation: use only one Gibbs step and compare the negative-phase statistics to a longer chain.
- Failure signal: D5 improves visually while the summary metric worsens, or one mode repeats across many generated panels.
- Reproducibility: keep the seed fixed, then change only one knob at a time.

Practice 1: Change the latent dimension or codebook size and predict which rung changes most.

Practice 2: Replace the D5 nuisance strength with a smaller value and compare the metric curve.

Practice 3: Add one diagnostic printout that would catch the named pitfall before plotting.